# Metal and self-sputtering

With a `metal` target, `DriftFluid2D` follows the sputtered atoms and their ions next to the
argon: the ions that hit the cathode sputter metal atoms with a yield that depends on their
energy, the atoms are ionized once ($\mathrm{M^+}$) and twice ($\mathrm{M^{2+}}$), and every
ion species emits its own number of secondary electrons. This page shows the yields, then
a short run of a copper target seeded with a metal-rich plasma: the metal fraction, the ion
composition at the target and the cathode current by species. All yields and rates of the
metal are placeholders of the right order (see `carafe.magnetron.MATERIALS`), the run is
coarse and lasts one microsecond.

In [ ]:
import numpy as np
import plasma_plots
import xarray as xr

from carafe import DriftFluid2D
from carafe.driftfluid2d import exponential_field
from carafe.magnetron import MATERIALS

plasma_plots.set_backend("plotly")
plasma_plots.set_math("latex");

## Yields of the ion energy

The sputter yields are Bohdansky's formula (`carafe.sputtering.BohdanskyYield`): the nuclear
stopping of the ion in the target times a threshold factor, zero below the threshold
$E_\mathrm{th}$ that the masses and the surface binding energy give. An ion of charge $Z$
falls through the cathode fall $V$ and arrives with $Z e V$, so a $\mathrm{Cu^{2+}}$ sputters
as a copper atom of twice the energy:

In [ ]:
copper = MATERIALS["Cu"]
fall = np.geomspace(10.0, 1000.0, 120)
ions = {"Ar+": 1, "Cu+": 1, "Cu2+": 2}
yields = xr.DataArray(
    [
        copper.sputter_yield("Ar+" if ion == "Ar+" else "M+", charge * fall)
        for ion, charge in ions.items()
    ],
    dims=("ion", "fall"),
    coords={
        "ion": list(ions),
        "fall": ("fall", fall, {"long_name": "cathode fall $V$", "units": "V"}),
    },
    attrs={"long_name": "sputter yield $Y(Z e V)$"},
)
yields.plasma.plot.profiles(x="fall", over="ion", logx=True)

In [ ]:
print(
    f"thresholds: Ar+ {copper.argon_yield.threshold:.1f} eV, "
    f"Cu {copper.self_yield.threshold:.1f} eV"
)
for name, target in MATERIALS.items():
    gammas = ", ".join(
        f"{ion} {target.secondary_yield(ion):.3f}" for ion in ("Ar+", "M+", "M2+")
    )
    print(f"{name}: secondary electrons per ion: {gammas}")

The secondary electron yields are Baragiola's potential emission,
$\gamma = 0.032\,(0.78\,E_\mathrm{iz} - 2W)$: an ion emits only if $0.78\,E_\mathrm{iz} > 2W$.
A singly charged metal ion on its own target never does; the doubly charged one, with the
second ionization energy, does. A discharge that runs on self-sputtering keeps its secondary
electrons only through the $\mathrm{M^{2+}}$ (Anders 2008).

## A short run of a copper target

A 2 cm gap at 400 V in 0.6 Pa of argon, the magnetic field of Boeuf's benchmark, 16 cells
along $x$ and 4 along $y$ (a 1D discharge). The start is a plasma of $10^{16}$ m$^{-3}$ whose
ions are half copper (a tenth of them doubly charged), in front of a cloud of copper atoms,
as a HiPIMS pulse would leave them. The sputtered atoms leave the cathode at 4 km/s (a copper
atom of 5 eV).

In [ ]:
gas = 0.6 / (1.380649e-23 * 300.0)  # m^-3 at 0.6 Pa, 300 K
model = DriftFluid2D(
    0.02,
    0.008,
    16,
    4,
    voltage=400.0,
    magnetic_field=exponential_field(0.12, 0.006),
    gas_density=gas,
    electron_temperature=None,
    metal=copper,
    metal_speed=4000.0,
)
x = model.x[:, None] * np.ones((1, model.ny))
density = 1e16
start = model.initial_state(
    0.5 * density,
    electron_temperature=3.0,
    metal_density=1e18 * np.exp(-x / 0.005),
    metal_ion_density=0.45 * density,
    metal_ion2_density=0.05 * density,
)
budgets = []
run = model.run(
    start,
    1e-6,
    cfl=0.2,
    record_every=5e-8,
    callback=lambda state: budgets.append(model.cathode_budget(state).mean("y")),
)
final = run.state
budgets.append(model.cathode_budget(final).mean("y"))
budget = xr.concat(budgets, dim="time").assign_coords(
    time=("time", 1e6 * np.array([*run.times]), {"long_name": "$t$", "units": "µs"})
)
run.steps

The metal fraction of the neutrals and of the ions along the gap at the end: the atoms
thin out from the cathode, where they are sputtered, towards the anode, and the metal ions
are lost to the cathode faster than the atoms are ionized:

In [ ]:
data = final.to_dataset().mean("y")
data = data.assign_coords(
    x=("x", 1e3 * data.x.values, {"long_name": "$x$", "units": "mm"})
)
metal_fraction = data.metal_density / (data.metal_density + data.gas_density)
metal_fraction.attrs["long_name"] = (
    r"metal fraction of the neutrals $n_\mathrm{M}/(n_\mathrm{M} + N)$"
)
ions = data.ion_density + data.metal_ion_density + data.metal_ion2_density
ion_fraction = xr.concat(
    [data.metal_ion_density / ions, data.metal_ion2_density / ions],
    dim=xr.DataArray(["Cu+", "Cu2+"], dims="ion"),
)
ion_fraction.attrs["long_name"] = "share of the ion density"
with plasma_plots.figure(2, 1, figsize=(7, 7)) as fig:
    metal_fraction.expand_dims(gas=["Cu in Ar"]).plasma.plot.profiles(
        x="x", over="gas", logy=True, ax=fig[0]
    )
    ion_fraction.plasma.plot.profiles(x="x", over="ion", ax=fig[1])

## At the target

The share of each species in the ion flux into the cathode, and the current density at the
cathode by species, $Z e \Gamma$ of the ions and $\gamma e \Gamma$ of their secondary
electrons. As the metal ions drain into the cathode and the argon ions multiply, the
argon takes over; the $\mathrm{Cu^{+}}$ add ion current but no electrons, the
$\mathrm{Cu^{2+}}$ both:

In [ ]:
species = {"Ar+": "Ar+", "M+": "Cu+", "M2+": "Cu2+"}
budget = budget.assign_coords(species=[species[s] for s in budget.species.values])
share = budget.ion_flux / budget.ion_flux.sum("species")
share.attrs["long_name"] = "share of the ion flux into the cathode"
parts = {
    f"{ion}: ions": budget.ion_current.sel(species=ion) for ion in species.values()
}
parts.update(
    {
        f"{ion}: electrons": budget.electron_current.sel(species=ion)
        for ion in ("Ar+", "Cu2+")
    }
)
current = xr.concat(list(parts.values()), dim=xr.DataArray(list(parts), dims="part"))
current.attrs.update(long_name="current density at the cathode", units="A/m^2")
with plasma_plots.figure(2, 1, figsize=(7, 7)) as fig:
    share.plasma.plot.profiles(x="time", over="species", ax=fig[0])
    current.plasma.plot.profiles(x="time", over="part", logy=True, ax=fig[1])

The cathode current density is the sum over the species of $(Z + \gamma)\, e\Gamma$, and the
self-sputtering share the part of the sputtered atoms that the copper ions sputter:

In [ ]:
end = budget.isel(time=-1)
total = float((end.ion_current + end.electron_current).sum())
print(
    f"j = {float(end.current_density):.1f} A/m^2 = sum (Z + gamma) e Gamma = {total:.1f} A/m^2"
)
print(
    f"self-sputtering share: {float(budget.self_sputtering_share[1]):.2f} -> {float(end.self_sputtering_share):.2f}"
)

## Options

The open choices of the metal are options of `DriftFluid2D`, with these defaults:

- `ion_energy="fall"`: the ions arrive with $Z e$ times the cathode fall (the largest potential
  above the cathode's); `"kinetic"` takes the energy of the ion fluid next to the cathode, a
  number a fixed fall;
- `double_ionization=True`: the $\mathrm{M^{2+}}$ fluid; `False` keeps $\mathrm{M^{+}}$ alone;
- `metal_collisions=True`: excitation (an energy loss) and elastic collisions (in the
  mobility) of the electrons on the atoms;
- `metal_transport="beam"`: the atoms cross the gap at `metal_speed`; `"diffusion"`
  thermalizes them at once (`metal_diffusion`);
- `argon_emission="material"`: $\mathrm{Ar^{+}}$ emits the material's yield; `"model"` keeps
  `secondary_emission`;
- `metal_ion_friction_rate=None`: the metal ions feel the argon ions' friction.